# ANKA-50M — C0 to R2/M2 Architecture Experiments

> **Status:** Architecture-search history

## Purpose

Build and compare the C0/C1/C2 controls and the first two generations of Reflect and Morph variants under matched parameter and token budgets.

## Execution guidance

Run setup and structural checks first. Training cells are independent experiment arms; run only the arm you intend to train, then execute the fair-comparison section after all required checkpoints exist.

## Primary outputs

Strict-load checks, pilot checkpoints, validation loss, throughput, and comparison reports.

## Reproducibility notes

- Code-cell outputs and execution counters are intentionally cleared in the public release.
- Seeds, persistent paths, and checkpoint-resume behavior remain in the code.
- Review path constants before execution; historical notebooks may reference earlier run directories.
- Execute sections from top to bottom unless a section explicitly describes an independent experiment arm.


## 1. Restore the run_01 project in a fresh Colab session

Prepares or validates persistent data artifacts. Review the Drive paths before running; large corpus operations can take time and may replace files at the configured destination.


In [ ]:
# ============================================================
# ANKA-50M — NEW COLAB SESSION SINGLE CELL SETUP
# ============================================================

from google.colab import drive
from pathlib import Path
import shutil
import subprocess
import sys
import json
import time
import torch

# ------------------------------------------------------------
# 1. MOUNT DRIVE
# ------------------------------------------------------------

drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/ANKA")
RUN_ROOT = DRIVE_ROOT / "run_01"

ZIP_PATH = DRIVE_ROOT / "ANKA-50M-full-pipeline.zip"
PROJECT_DIR = Path("/content/ANKA-50M")

DRIVE_SHARDS = RUN_ROOT / "shards"
LOCAL_SHARDS = Path("/content/anka_shards")

TOKENIZER_PATH = RUN_ROOT / "tokenizer" / "tokenizer.json"
MANIFEST_PATH = LOCAL_SHARDS / "manifest.json"

# ------------------------------------------------------------
# 2. VERIFY REQUIRED FILES
# ------------------------------------------------------------

assert ZIP_PATH.exists(), f"Project ZIP not found: {ZIP_PATH}"
assert DRIVE_SHARDS.exists(), f"Shard folder not found: {DRIVE_SHARDS}"
assert TOKENIZER_PATH.exists(), f"Tokenizer not found: {TOKENIZER_PATH}"
assert (DRIVE_SHARDS / "manifest.json").exists(), "Drive manifest.json not found"

# ------------------------------------------------------------
# 3. EXTRACT PROJECT
# ------------------------------------------------------------

if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)

PROJECT_DIR.mkdir(parents=True)

subprocess.run(
    [
        "unzip",
        "-q",
        str(ZIP_PATH),
        "-d",
        str(PROJECT_DIR),
    ],
    check=True,
)

# ------------------------------------------------------------
# 4. INSTALL DEPENDENCIES
# ------------------------------------------------------------

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-r",
        str(PROJECT_DIR / "requirements.txt"),
    ],
    check=True,
)

# ------------------------------------------------------------
# 5. COPY SHARDS TO LOCAL DISK
# ------------------------------------------------------------

if LOCAL_SHARDS.exists():
    shutil.rmtree(LOCAL_SHARDS)

print("Copying shards from Drive to local disk...")
copy_start = time.time()

shutil.copytree(DRIVE_SHARDS, LOCAL_SHARDS)

copy_minutes = (time.time() - copy_start) / 60

# ------------------------------------------------------------
# 6. DATA VALIDATION
# ------------------------------------------------------------

binary_files = sorted(LOCAL_SHARDS.glob("*.bin"))

with MANIFEST_PATH.open("r", encoding="utf-8") as file:
    manifest = json.load(file)

train_shards = [
    item for item in manifest["shards"]
    if item["split"] == "train"
]

validation_shards = [
    item for item in manifest["shards"]
    if item["split"] == "validation"
]

train_tokens = sum(item["tokens"] for item in train_shards)
validation_tokens = sum(
    item["tokens"] for item in validation_shards
)

assert len(binary_files) == 138, (
    f"138 binary shards expected, found: {len(binary_files)}"
)

assert manifest["vocab_size"] == 15_360
assert manifest["total_tokens"] == 3_418_130_380

# ------------------------------------------------------------
# 7. GPU VALIDATION
# ------------------------------------------------------------

assert torch.cuda.is_available(), "CUDA GPU not found"

gpu_name = torch.cuda.get_device_name(0)
gpu_memory = (
    torch.cuda.get_device_properties(0).total_memory / 2**30
)

assert torch.cuda.is_bf16_supported(), (
    "Selected GPU does not support BF16"
)

# ------------------------------------------------------------
# 8. CONCLUSION
# ------------------------------------------------------------

print("\n============================================")
print("ANKA-50M SESSION READY")
print("============================================")
print("GPU                 :", gpu_name)
print(f"GPU VRAM            : {gpu_memory:.2f} GB")
print("BF16                : Supported")
print("Project             :", PROJECT_DIR)
print("Tokenizer           :", TOKENIZER_PATH)
print("Local manifest      :", MANIFEST_PATH)
print("Train shard         :", len(train_shards))
print("Validation shard    :", len(validation_shards))
print(f"Train token         : {train_tokens:,}")
print(f"Validation token    : {validation_tokens:,}")
print(f"Total token         : {manifest['total_tokens']:,}")
print(f"Copy time           : {copy_minutes:.2f} minutes")
print("Status              : READY FOR C0 TRAINING")

## 2. Stage binary shards on local storage

Prepares or validates persistent data artifacts. Review the Drive paths before running; large corpus operations can take time and may replace files at the configured destination.


In [ ]:
from pathlib import Path
import shutil
import time

SOURCE = Path("/content/drive/MyDrive/ANKA/run_01/shards")
TARGET = Path("/content/anka_shards")

if TARGET.exists():
    shutil.rmtree(TARGET)

start = time.time()
shutil.copytree(SOURCE, TARGET)

print(f"Copy completed: {(time.time() - start) / 60:.1f} minutes")
print("File count:", len(list(TARGET.glob("*.bin"))))
print("Manifest:", (TARGET / "manifest.json").exists())

## 3. Define C0, C1, C2, R1, and M1 variants

This is a compute-intensive stage. Confirm the selected runtime, input paths, token budget, and resume directory before execution. Checkpoints are written incrementally.


In [ ]:
%%writefile /content/ANKA-50M/anka/variants.py

"""
ANKA-50M controlled experiment models.

C0: standard 28x384 baseline
C1: C0 + residual-aware initialization
C2: Wider and shallower 20x448 model
R1: C0 + learnable residual Reflect gates
M1: C0 + token/layer conditioned Morph router

All models use the shared ANKAForCausalLM backbone.
This keeps attention, RoPE, loss, and training behavior
identical across experiments; only the specified experimental variable changes.
"""

from __future__ import annotations

from typing import Type

from .model import ANKAConfig, ANKAForCausalLM


# ============================================================
# 1. C0 — BASELINE
# ============================================================

def c0_config(
    vocab_size: int = 15_360,
    max_seq_len: int = 1_024,
) -> ANKAConfig:
    """
    Standard ANKA-50M control model.

    architecture:
        layers       = 28
        hidden       = 384
        heads        = 6
        kv_heads     = 2
        intermediate = 1024
        initialization = standard Normal(0, 0.02)
    """

    return ANKAConfig(
        vocab_size=vocab_size,
        max_seq_len=max_seq_len,
        n_layers=28,
        d_model=384,
        n_heads=6,
        n_kv_heads=2,
        d_ff=1_024,
        rope_theta=10_000.0,
        rms_norm_eps=1e-6,
        dropout=0.0,
        init_std=0.02,
        residual_scaled_init=False,
        use_reflect=False,
        use_morph=False,
    )


class C0BaselineModel(ANKAForCausalLM):
    """standard 28x384 ANKA-50M baseline model."""

    def __init__(
        self,
        vocab_size: int = 15_360,
        max_seq_len: int = 1_024,
    ) -> None:
        super().__init__(
            c0_config(
                vocab_size=vocab_size,
                max_seq_len=max_seq_len,
            )
        )


# ============================================================
# 2. C1 — RESIDUAL-AWARE INITIALIZATION
# ============================================================

def c1_config(
    vocab_size: int = 15_360,
    max_seq_len: int = 1_024,
) -> ANKAConfig:
    """
    Uses the same architecture as C0.

    Only change:
        attention.o_proj and FFN down_proj initial standard
        deviation is scaled by layer count.

        residual_std = init_std / sqrt(2 * n_layers)

    Parameter count is identical to C0.
    """

    return ANKAConfig(
        vocab_size=vocab_size,
        max_seq_len=max_seq_len,
        n_layers=28,
        d_model=384,
        n_heads=6,
        n_kv_heads=2,
        d_ff=1_024,
        rope_theta=10_000.0,
        rms_norm_eps=1e-6,
        dropout=0.0,
        init_std=0.02,
        residual_scaled_init=True,
        use_reflect=False,
        use_morph=False,
    )


class C1ResidualInitModel(ANKAForCausalLM):
    """C1 model using residual-aware initialization."""

    def __init__(
        self,
        vocab_size: int = 15_360,
        max_seq_len: int = 1_024,
    ) -> None:
        super().__init__(
            c1_config(
                vocab_size=vocab_size,
                max_seq_len=max_seq_len,
            )
        )


# ============================================================
# 3. C2 — WIDE / SHALLOW
# ============================================================

def c2_config(
    vocab_size: int = 15_360,
    max_seq_len: int = 1_024,
) -> ANKAConfig:
    """
    Wider and shallower check model.

    architecture:
        layers       = 20
        hidden       = 448
        heads        = 7
        kv_heads     = 1
        head_dim     = 64
        intermediate = 1216

    Goal:
        28x384 deep/dar tasarim with 20x448 wide/short
        tasarimi same under approximately the same parameter budget.
    """

    return ANKAConfig(
        vocab_size=vocab_size,
        max_seq_len=max_seq_len,
        n_layers=20,
        d_model=448,
        n_heads=7,
        n_kv_heads=1,
        d_ff=1_216,
        rope_theta=10_000.0,
        rms_norm_eps=1e-6,
        dropout=0.0,
        init_std=0.02,
        residual_scaled_init=False,
        use_reflect=False,
        use_morph=False,
    )


class C2WideShallowModel(ANKAForCausalLM):
    """20-layer C2 model with hidden width 448."""

    def __init__(
        self,
        vocab_size: int = 15_360,
        max_seq_len: int = 1_024,
    ) -> None:
        super().__init__(
            c2_config(
                vocab_size=vocab_size,
                max_seq_len=max_seq_len,
            )
        )


# ============================================================
# 4. R1 — ANKA-REFLECT
# ============================================================

def r1_config(
    vocab_size: int = 15_360,
    max_seq_len: int = 1_024,
) -> ANKAConfig:
    """
    First ANKA-Reflect model.

    Adds two learnable controls to each Transformer layer in the C0 backbone.
    residual gate eklenir:

        attention residual gate
        FFN residual gate

    Gate initialization probability:
        0.95

    This keeps the model close to the baseline at the start of training,
    then each in-layer residual correction learns the update magnitude.
    """

    return ANKAConfig(
        vocab_size=vocab_size,
        max_seq_len=max_seq_len,
        n_layers=28,
        d_model=384,
        n_heads=6,
        n_kv_heads=2,
        d_ff=1_024,
        rope_theta=10_000.0,
        rms_norm_eps=1e-6,
        dropout=0.0,
        init_std=0.02,
        residual_scaled_init=False,
        use_reflect=True,
        reflect_base_probability=0.95,
        use_morph=False,
    )


class R1ReflectModel(ANKAForCausalLM):
    """R1 model using a learnable residual gate."""

    def __init__(
        self,
        vocab_size: int = 15_360,
        max_seq_len: int = 1_024,
    ) -> None:
        super().__init__(
            r1_config(
                vocab_size=vocab_size,
                max_seq_len=max_seq_len,
            )
        )


# ============================================================
# 5. M1 — ANKA-MORPH
# ============================================================

def m1_config(
    vocab_size: int = 15_360,
    max_seq_len: int = 1_024,
) -> ANKAConfig:
    """
    First ANKA-Morph model.

    shared small router, token representation and layer kimligine by
    mixes three activation behaviors:

        1. SiLU(gate) * value
        2. gate * value
        3. tanh(gate) * value

    The standard SwiGLU branch starts with probability 0.98.
    This keeps M1 close to C0 at the start of training.
    """

    return ANKAConfig(
        vocab_size=vocab_size,
        max_seq_len=max_seq_len,
        n_layers=28,
        d_model=384,
        n_heads=6,
        n_kv_heads=2,
        d_ff=1_024,
        rope_theta=10_000.0,
        rms_norm_eps=1e-6,
        dropout=0.0,
        init_std=0.02,
        residual_scaled_init=False,
        use_reflect=False,
        use_morph=True,
        morph_width=16,
        morph_base_probability=0.98,
        morph_router_init_std=1e-3,
    )


class M1MorphModel(ANKAForCausalLM):
    """M1 model using a token/layer-conditioned Morph router."""

    def __init__(
        self,
        vocab_size: int = 15_360,
        max_seq_len: int = 1_024,
    ) -> None:
        super().__init__(
            m1_config(
                vocab_size=vocab_size,
                max_seq_len=max_seq_len,
            )
        )


# ============================================================
# 6. EXPERIMENT API
# ============================================================

EXPERIMENT_ARMS = (
    "C0",
    "C1",
    "C2",
    "R1",
    "M1",
)


MODEL_CLASSES: dict[str, Type[ANKAForCausalLM]] = {
    "C0": C0BaselineModel,
    "C1": C1ResidualInitModel,
    "C2": C2WideShallowModel,
    "R1": R1ReflectModel,
    "M1": M1MorphModel,
}


def experiment_config(
    name: str,
    *,
    vocab_size: int = 15_360,
    max_seq_len: int = 1_024,
) -> ANKAConfig:
    """
    Configuration entry point used by the existing training system.
    """

    key = name.upper()

    builders = {
        "C0": c0_config,
        "C1": c1_config,
        "C2": c2_config,
        "R1": r1_config,
        "M1": m1_config,
    }

    if key not in builders:
        raise KeyError(
            f"Bilinmeyen experiment kolu: {name}. "
            f"Valid models: {EXPERIMENT_ARMS}"
        )

    return builders[key](
        vocab_size=vocab_size,
        max_seq_len=max_seq_len,
    )


def build_experiment_model(
    name: str,
    *,
    vocab_size: int = 15_360,
    max_seq_len: int = 1_024,
) -> ANKAForCausalLM:
    """
    Builds the requested concrete model class.
    """

    key = name.upper()

    if key not in MODEL_CLASSES:
        raise KeyError(
            f"Bilinmeyen experiment kolu: {name}. "
            f"Valid models: {EXPERIMENT_ARMS}"
        )

    return MODEL_CLASSES[key](
        vocab_size=vocab_size,
        max_seq_len=max_seq_len,
    )

## 4. Verify the five first-generation models

This is a compute-intensive stage. Confirm the selected runtime, input paths, token budget, and resume directory before execution. Checkpoints are written incrementally.


In [ ]:
# ============================================================
# VERIFY FIVE MODELS
# ============================================================

%cd /content/ANKA-50M

import gc
import torch

from anka.variants import (
    EXPERIMENT_ARMS,
    build_experiment_model,
)

expected_parameters = {
    "C0": 49_960_320,
    "C1": 49_960_320,
    "C2": 48_760_768,
    "R1": 49_960_376,
    "M1": 49_966_963,
}

for arm in EXPERIMENT_ARMS:
    model = build_experiment_model(arm)

    actual = model.num_trainable_parameters()
    expected = expected_parameters[arm]

    assert actual == expected, (
        f"{arm} parameter mismatch: "
        f"expected={expected:,}, actual={actual:,}"
    )

    config = model.config

    print("\n" + "=" * 50)
    print("Model          :", arm)
    print("Class          :", model.__class__.__name__)
    print("parameter      :", f"{actual:,}")
    print("layer         :", config.n_layers)
    print("Hidden         :", config.d_model)
    print("Head / KV      :", config.n_heads, "/", config.n_kv_heads)
    print("FFN            :", config.d_ff)
    print("Residual init  :", config.residual_scaled_init)
    print("Reflect        :", config.use_reflect)
    print("Morph          :", config.use_morph)

    del model
    gc.collect()

print("\n✓ C0, C1, C2, R1 and M1 models were all verified.")

## 5. Persist the first-generation variant definitions

Initializes the runtime, deterministic settings, dependencies, and persistent paths required by later sections.


In [ ]:
from pathlib import Path
import shutil

source = Path("/content/ANKA-50M/anka/variants.py")

target = Path(
    "/content/drive/MyDrive/ANKA/run_01/code/"
    "anka/variants.py"
)

target.parent.mkdir(parents=True, exist_ok=True)

shutil.copy2(source, target)

print("saved:", target)

## 6. Train C1 with residual-aware initialization

This is a compute-intensive stage. Confirm the selected runtime, input paths, token budget, and resume directory before execution. Checkpoints are written incrementally.


In [ ]:
# ============================================================
# C1 — CLEAN AND DIRECT START
# ============================================================

%cd /content/ANKA-50M

# Kill background C1 training process if any.
!pkill -f "anka.experiment train --arm C1" || true

import time
import torch

time.sleep(3)

torch.cuda.empty_cache()
torch.cuda.synchronize()

print("Clean start VRAM:")
!nvidia-smi --query-compute-apps=pid,used_memory --format=csv

# Run directly by preventing log buffering.
!PYTHONUNBUFFERED=1 python -u -m anka.experiment train \
  --arm C1 \
  --manifest /content/anka_shards/manifest.json \
  --tokenizer /content/drive/MyDrive/ANKA/run_01/tokenizer/tokenizer.json \
  --output /content/drive/MyDrive/ANKA/run_01/pilot \
  --token-budget 256114688 \
  --global-batch-tokens 262144 \
  --micro-batch-size 64 \
  --eval-batch-size 8 \
  --sequence-length 1024 \
  --max-seq-len 1024 \
  --vocab-size 15360 \
  --learning-rate 0.003 \
  --min-learning-rate 0.0003 \
  --warmup-fraction 0.05 \
  --precision bf16 \
  --device cuda \
  --seed 2026 \
  --log-every 10 \
  --eval-every 100 \
  --eval-batches 20 \
  --prompts "The future of" "One day in Istanbul"

## 7. Train the wider and shallower C2 control

This is a compute-intensive stage. Confirm the selected runtime, input paths, token budget, and resume directory before execution. Checkpoints are written incrementally.


In [ ]:
# ============================================================
# C2 — 20 layer × 448 HIDDEN
# ============================================================

%cd /content/ANKA-50M

!PYTHONUNBUFFERED=1 python -u -m anka.experiment train \
  --arm C2 \
  --manifest /content/anka_shards/manifest.json \
  --tokenizer /content/drive/MyDrive/ANKA/run_01/tokenizer/tokenizer.json \
  --output /content/drive/MyDrive/ANKA/run_01/pilot \
  --token-budget 256114688 \
  --global-batch-tokens 262144 \
  --micro-batch-size 64 \
  --eval-batch-size 8 \
  --sequence-length 1024 \
  --max-seq-len 1024 \
  --vocab-size 15360 \
  --learning-rate 0.003 \
  --min-learning-rate 0.0003 \
  --warmup-fraction 0.05 \
  --precision bf16 \
  --device cuda \
  --seed 2026 \
  --log-every 10 \
  --eval-every 100 \
  --eval-batches 20 \
  --prompts "The future of" "One day in Istanbul"

## 8. Train the first-generation R1 Reflect variant

This is a compute-intensive stage. Confirm the selected runtime, input paths, token budget, and resume directory before execution. Checkpoints are written incrementally.


In [ ]:
# ============================================================
# R1 — LEARNABLE RESIDUAL REFLECT GATE
# ============================================================

%cd /content/ANKA-50M

!PYTHONUNBUFFERED=1 python -u -m anka.experiment train \
  --arm R1 \
  --manifest /content/anka_shards/manifest.json \
  --tokenizer /content/drive/MyDrive/ANKA/run_01/tokenizer/tokenizer.json \
  --output /content/drive/MyDrive/ANKA/run_01/pilot \
  --token-budget 256114688 \
  --global-batch-tokens 262144 \
  --micro-batch-size 64 \
  --eval-batch-size 8 \
  --sequence-length 1024 \
  --max-seq-len 1024 \
  --vocab-size 15360 \
  --learning-rate 0.003 \
  --min-learning-rate 0.0003 \
  --warmup-fraction 0.05 \
  --precision bf16 \
  --device cuda \
  --seed 2026 \
  --log-every 10 \
  --eval-every 100 \
  --eval-batches 20 \
  --prompts "The future of" "One day in Istanbul"

## 9. Train the first-generation M1 Morph variant

This is a compute-intensive stage. Confirm the selected runtime, input paths, token budget, and resume directory before execution. Checkpoints are written incrementally.


In [ ]:
# ============================================================
# M1 — TOKEN/LAYER-CONDITIONED MORPH ROUTER
# ============================================================

%cd /content/ANKA-50M

!PYTHONUNBUFFERED=1 python -u -m anka.experiment train \
  --arm M1 \
  --manifest /content/anka_shards/manifest.json \
  --tokenizer /content/drive/MyDrive/ANKA/run_01/tokenizer/tokenizer.json \
  --output /content/drive/MyDrive/ANKA/run_01/pilot \
  --token-budget 256114688 \
  --global-batch-tokens 262144 \
  --micro-batch-size 64 \
  --eval-batch-size 8 \
  --sequence-length 1024 \
  --max-seq-len 1024 \
  --vocab-size 15360 \
  --learning-rate 0.003 \
  --min-learning-rate 0.0003 \
  --warmup-fraction 0.05 \
  --precision bf16 \
  --device cuda \
  --seed 2026 \
  --log-every 10 \
  --eval-every 100 \
  --eval-batches 20 \
  --prompts "The future of" "One day in Istanbul"

## 10. Evaluate all first-generation pilot checkpoints

Defines or validates model components used by the experiment. Keep tensor shapes, parameter counts, and causal behavior unchanged when reproducing the run.


In [ ]:
# ============================================================
# ANKA-50M — EVALUATE ALL PILOT MODELS TOGETHER
# ============================================================

%cd /content/ANKA-50M

from pathlib import Path
import csv
import gc
import json
import math
import statistics
import time

import torch

from anka.data import ShardBatchIterator
from anka.model import ANKAConfig, ANKAForCausalLM
from tokenizers import Tokenizer


# ============================================================
# 1. SETTINGS
# ============================================================

PILOT_ROOT = Path(
    "/content/drive/MyDrive/ANKA/run_01/pilot"
)

MANIFEST_PATH = Path(
    "/content/anka_shards/manifest.json"
)

TOKENIZER_PATH = Path(
    "/content/drive/MyDrive/ANKA/run_01/"
    "tokenizer/tokenizer.json"
)

REPORT_ROOT = Path(
    "/content/drive/MyDrive/ANKA/run_01/"
    "pilot_comparison"
)

REPORT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

ARMS = [
    "C0",
    "C1",
    "C2",
    "R1",
    "M1",
]

EVAL_SEED = 98_761
EVAL_BATCH_SIZE = 8
EVAL_BATCHES = 100
SEQUENCE_LENGTH = 1_024
MAX_NEW_TOKENS = 64

PROMPTS = [
    "The future of artificial intelligence",
    "Scientists discovered that",
    "The city was quiet until",
    "One day in Istanbul",
    "The future of artificial intelligence",
]

DEVICE = torch.device("cuda")

assert torch.cuda.is_available()
assert MANIFEST_PATH.exists()
assert TOKENIZER_PATH.exists()


# ============================================================
# 2. TOKENIZER
# ============================================================

tokenizer = Tokenizer.from_file(
    str(TOKENIZER_PATH)
)

eos_id = tokenizer.token_to_id("<eos>")
bos_id = tokenizer.token_to_id("<empty>")


# ============================================================
# 3. CHECKPOINT LOADING
# ============================================================

def load_model(
    checkpoint_path: Path,
) -> tuple[ANKAForCausalLM, dict]:

    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False,
    )

    config = ANKAConfig(
        **checkpoint["model_config"]
    )

    model = ANKAForCausalLM(
        config
    )

    model.load_state_dict(
        checkpoint["model"],
        strict=True,
    )

    model = model.to(DEVICE)
    model.eval()

    return model, checkpoint


# ============================================================
# 4. FIXED VALIDATION TEST
# ============================================================

@torch.inference_mode()
def evaluate_model(
    model: ANKAForCausalLM,
) -> dict:

    # Use the same seed for every model:
    # select the same shards and token offsets.
    validation_iterator = iter(
        ShardBatchIterator(
            manifest_path=MANIFEST_PATH,
            split="validation",
            batch_size=EVAL_BATCH_SIZE,
            sequence_length=SEQUENCE_LENGTH,
            seed=EVAL_SEED,
        )
    )

    losses = []
    total_tokens = 0

    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()

    start_time = time.perf_counter()

    for _ in range(EVAL_BATCHES):

        tokens = next(
            validation_iterator
        ).to(
            DEVICE,
            dtype=torch.long,
            non_blocking=True,
        )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.bfloat16,
        ):
            output = model(
                tokens,
                targets=tokens,
            )

        loss = float(
            output.loss.detach().float()
        )

        losses.append(loss)
        total_tokens += tokens.numel()

        del output
        del tokens

    torch.cuda.synchronize()

    elapsed_seconds = (
        time.perf_counter()
        - start_time
    )

    mean_loss = statistics.fmean(
        losses
    )

    loss_std = (
        statistics.stdev(losses)
        if len(losses) > 1
        else 0.0
    )

    perplexity = math.exp(
        min(mean_loss, 20.0)
    )

    tokens_per_second = (
        total_tokens
        / elapsed_seconds
    )

    peak_memory_gib = (
        torch.cuda.max_memory_allocated()
        / 2**30
    )

    return {
        "validation_loss": mean_loss,
        "validation_loss_std": loss_std,
        "perplexity": perplexity,
        "eval_tokens": total_tokens,
        "eval_seconds": elapsed_seconds,
        "eval_tokens_per_second": tokens_per_second,
        "eval_peak_memory_gib": peak_memory_gib,
    }


# ============================================================
# 5. FIXED GREEDY GENERATION
# ============================================================

@torch.inference_mode()
def generate_text(
    model: ANKAForCausalLM,
    prompt: str,
) -> str:

    prompt_ids = tokenizer.encode(
        prompt
    ).ids

    if not prompt_ids:
        prompt_ids = [bos_id]

    tokens = torch.tensor(
        [prompt_ids],
        dtype=torch.long,
        device=DEVICE,
    )

    for _ in range(MAX_NEW_TOKENS):

        model_input = tokens[
            :,
            -model.config.max_seq_len:
        ]

        with torch.autocast(
            device_type="cuda",
            dtype=torch.bfloat16,
        ):
            output = model(
                model_input
            )

        next_token = torch.argmax(
            output.logits[:, -1, :].float(),
            dim=-1,
            keepdim=True,
        )

        tokens = torch.cat(
            [tokens, next_token],
            dim=1,
        )

        del output

        if (
            eos_id is not None
            and next_token.item() == eos_id
        ):
            break

    return tokenizer.decode(
        tokens[0].tolist()
    )


# ============================================================
# 6. TEST ALL MODELS
# ============================================================

results = []

for arm in ARMS:

    checkpoint_path = (
        PILOT_ROOT
        / arm
        / "checkpoint.pt"
    )

    result_path = (
        PILOT_ROOT
        / arm
        / "result.json"
    )

    print("\n" + "=" * 70)
    print(f"{arm} CHECKING")
    print("=" * 70)

    if not checkpoint_path.exists():

        print(
            f"{arm} checkpoint not found, "
            "skipping this model."
        )

        continue

    model, checkpoint = load_model(
        checkpoint_path
    )

    parameters = (
        model.num_trainable_parameters()
    )

    print(
        "parameter:",
        f"{parameters:,}",
    )

    evaluation = evaluate_model(
        model
    )

    generations = {}

    for prompt in PROMPTS:

        generated = generate_text(
            model,
            prompt,
        )

        generations[prompt] = generated

    original_result = {}

    if result_path.exists():

        with result_path.open(
            "r",
            encoding="utf-8",
        ) as file:
            original_result = json.load(file)

    row = {
        "arm": arm,
        "parameters": parameters,
        "training_token_budget": checkpoint[
            "train_state"
        ]["tokens_seen"],
        "original_validation_loss": (
            original_result.get(
                "validation_loss"
            )
        ),
        **evaluation,
        "generations": generations,
    }

    results.append(row)

    print(
        "fixed validation loss :",
        f"{evaluation['validation_loss']:.6f}",
    )

    print(
        "Perplexity            :",
        f"{evaluation['perplexity']:.3f}",
    )

    print(
        "Eval token/s          :",
        f"{evaluation['eval_tokens_per_second']:,.0f}",
    )

    print(
        "Peak VRAM             :",
        f"{evaluation['eval_peak_memory_gib']:.2f} GB",
    )

    for prompt, generated in generations.items():

        print("\nPrompt:", prompt)
        print("Output :", generated)

    del model
    del checkpoint

    gc.collect()
    torch.cuda.empty_cache()


# ============================================================
# 7. ranking
# ============================================================

results.sort(
    key=lambda item: (
        item["validation_loss"],
        -item["eval_tokens_per_second"],
    )
)

print("\n" + "=" * 90)
print("ANKA-50M PILOT COMPARISON")
print("=" * 90)

for rank, result in enumerate(
    results,
    start=1,
):

    print(
        f"{rank}. "
        f"{result['arm']} | "
        f"loss={result['validation_loss']:.6f} | "
        f"ppl={result['perplexity']:.2f} | "
        f"eval={result['eval_tokens_per_second']:,.0f} tok/s | "
        f"VRAM={result['eval_peak_memory_gib']:.2f} GB"
    )


# ============================================================
# 8. JSON record
# ============================================================

json_path = (
    REPORT_ROOT
    / "pilot_comparison.json"
)

with json_path.open(
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        {
            "evaluation_seed": EVAL_SEED,
            "evaluation_batches": EVAL_BATCHES,
            "evaluation_batch_size": EVAL_BATCH_SIZE,
            "sequence_length": SEQUENCE_LENGTH,
            "ranking": results,
        },
        file,
        ensure_ascii=False,
        indent=2,
    )


# ============================================================
# 9. CSV record
# ============================================================

csv_path = (
    REPORT_ROOT
    / "pilot_comparison.csv"
)

csv_columns = [
    "arm",
    "parameters",
    "training_token_budget",
    "original_validation_loss",
    "validation_loss",
    "validation_loss_std",
    "perplexity",
    "eval_tokens",
    "eval_seconds",
    "eval_tokens_per_second",
    "eval_peak_memory_gib",
]

with csv_path.open(
    "w",
    encoding="utf-8",
    newline="",
) as file:

    writer = csv.DictWriter(
        file,
        fieldnames=csv_columns,
    )

    writer.writeheader()

    for result in results:

        writer.writerow({
            column: result[column]
            for column in csv_columns
        })


print("\nRaporlar saved:")
print(json_path)
print(csv_path)

## 11. Define the second-generation R2 and M2 architectures

Defines or validates model components used by the experiment. Keep tensor shapes, parameter counts, and causal behavior unchanged when reproducing the run.


In [ ]:
%%writefile /content/ANKA-50M/anka/advanced_variants.py
"""
ANKA-50M second-generation experimental models.

R2:
    Token-dependent reflection/correction mechanism.
    Selected Transformer blocks run a second time with shared weights.
    A controller selects how strongly the second correction is applied to each token.

M2:
    Token- and layer-conditioned low-rank expert transforms.
    For each token, the router selects the two strongest experts from four candidates.
    Experts are shared across layers.

Both models use the strong 20×448 C2 backbone.
"""

from __future__ import annotations

import math
from typing import Optional

import torch
import torch.nn as nn
import torch.nn.functional as F

from .model import (
    ANKAConfig,
    ANKAForCausalLM,
    ANKAOutput,
    ANKABlock,
    RMSNorm,
)


# ============================================================
# 1. SHARED C2 BACKBONE
# ============================================================

def advanced_c2_config(
    vocab_size: int = 15_360,
    max_seq_len: int = 1_024,
) -> ANKAConfig:
    """Strong C2 backbone shared by R2 and M2."""

    return ANKAConfig(
        vocab_size=vocab_size,
        max_seq_len=max_seq_len,
        n_layers=20,
        d_model=448,
        n_heads=7,
        n_kv_heads=1,
        d_ff=1_216,
        rope_theta=10_000.0,
        rms_norm_eps=1e-6,
        dropout=0.0,
        init_std=0.02,
        residual_scaled_init=False,
        use_reflect=False,
        use_morph=False,
    )


class AdvancedANKABase(ANKAForCausalLM):
    """
    Ozel R2/M2 models for shared temel.

    ANKAForCausalLM'in yardimci fonksiyonlarini uses but
    The model is built here so specialized modules exist before the base model's initial
    It can be added before the parameter check.
    """

    def __init__(self, config: ANKAConfig) -> None:
        nn.Module.__init__(self)

        self.config = config

        self.token_embedding = nn.Embedding(
            config.vocab_size,
            config.d_model,
        )

        self.layers = nn.ModuleList(
            ANKABlock(config)
            for _ in range(config.n_layers)
        )

        self.morph_router = None

        self.final_norm = RMSNorm(
            config.d_model,
            eps=config.rms_norm_eps,
            affine=True,
        )

    def initialize_base_parameters(self) -> None:
        """Initializes standard in-model modules using the ANKA scheme."""

        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.normal_(
                    module.weight,
                    mean=0.0,
                    std=self.config.init_std,
                )

                if module.bias is not None:
                    nn.init.zeros_(module.bias)

            elif isinstance(module, nn.Embedding):
                nn.init.normal_(
                    module.weight,
                    mean=0.0,
                    std=self.config.init_std,
                )

            elif isinstance(module, RMSNorm):
                if module.weight is not None:
                    nn.init.ones_(module.weight)

    def verify_parameter_limit(self) -> None:
        """50M competition limit check does."""

        parameters = self.num_trainable_parameters()

        if parameters > self.PARAMETER_LIMIT:
            raise RuntimeError(
                f"Model exceeds the parameter limit: "
                f"{parameters:,} > {self.PARAMETER_LIMIT:,}"
            )

    def validate_inputs(
        self,
        input_ids: torch.Tensor,
        targets: Optional[torch.Tensor],
    ) -> None:
        if input_ids.ndim != 2:
            raise ValueError(
                "input_ids [batch, sequence] biciminde must-be"
            )

        if input_ids.dtype != torch.long:
            raise TypeError("input_ids must be torch.long")

        if input_ids.size(1) > self.config.max_seq_len:
            raise ValueError(
                f"Sequence length {input_ids.size(1)}, "
                f"model limit {self.config.max_seq_len}"
            )

        if targets is not None:
            if targets.shape != input_ids.shape:
                raise ValueError(
                    "targets and input_ids same sekilde must-be"
                )

            if targets.dtype != torch.long:
                raise TypeError("targets must be torch.long")

    def causal_loss(
        self,
        logits: torch.Tensor,
        targets: Optional[torch.Tensor],
    ) -> Optional[torch.Tensor]:
        if targets is None:
            return None

        if targets.size(1) < 2:
            raise ValueError(
                "Loss for most few two token required"
            )

        shift_logits = logits[:, :-1, :].contiguous()
        shift_targets = targets[:, 1:].contiguous()

        return F.cross_entropy(
            shift_logits.view(-1, self.config.vocab_size),
            shift_targets.view(-1),
            ignore_index=-100,
        )


# ============================================================
# 2. R2 — ANKA REFLECT
# ============================================================

class ReflectController(nn.Module):
    """
    First passin representation with examines its update.

    Input:
        Proposal: output of the first block pass
        delta: proposal - previous representation

    Output:
        For each token 0-1 arasinda correction gate
    """

    def __init__(
        self,
        d_model: int,
        hidden_width: int,
        eps: float,
        initial_probability: float,
    ) -> None:
        super().__init__()

        self.proposal_norm = RMSNorm(
            d_model,
            eps=eps,
            affine=False,
        )

        self.delta_norm = RMSNorm(
            d_model,
            eps=eps,
            affine=False,
        )

        self.input_projection = nn.Linear(
            2 * d_model,
            hidden_width,
            bias=False,
        )

        self.output_projection = nn.Linear(
            hidden_width,
            1,
            bias=True,
        )

        self.initial_probability = initial_probability

    def reset_special_parameters(self) -> None:
        """
        Keeps the second pass small at the beginning of training.

        gate is not exactly zero; dolayisiyla Reflect path
        from the first step can receive gradients.
        """

        nn.init.normal_(
            self.input_projection.weight,
            mean=0.0,
            std=0.02,
        )

        nn.init.normal_(
            self.output_projection.weight,
            mean=0.0,
            std=1e-3,
        )

        initial_logit = math.log(
            self.initial_probability
            / (1.0 - self.initial_probability)
        )

        nn.init.constant_(
            self.output_projection.bias,
            initial_logit,
        )

    def forward(
        self,
        proposal: torch.Tensor,
        delta: torch.Tensor,
    ) -> torch.Tensor:
        features = torch.cat(
            (
                self.proposal_norm(proposal),
                self.delta_norm(delta),
            ),
            dim=-1,
        )

        hidden = F.silu(
            self.input_projection(features)
        )

        gate_logits = self.output_projection(hidden)

        return torch.sigmoid(
            gate_logits.float()
        ).to(dtype=proposal.dtype)


class R2ReflectModel(AdvancedANKABase):
    """
    Second-generation ANKA-Reflect.

    In selected layers:

        before
        → same Transformer blogunun first passi
        → proposal
        → second pass with the same weights
        → correction_candidate
        → token-dependent gate
        → proposal + gate * correction

    The second pass does not create new Transformer parameters.
    """

    REFLECT_LAYERS = (4, 9, 14, 19)

    def __init__(
        self,
        vocab_size: int = 15_360,
        max_seq_len: int = 1_024,
    ) -> None:
        config = advanced_c2_config(
            vocab_size=vocab_size,
            max_seq_len=max_seq_len,
        )

        super().__init__(config)

        self.reflect_controllers = nn.ModuleDict(
            {
                str(layer_id): ReflectController(
                    d_model=config.d_model,
                    hidden_width=32,
                    eps=config.rms_norm_eps,
                    initial_probability=0.05,
                )
                for layer_id in self.REFLECT_LAYERS
            }
        )

        self.initialize_base_parameters()

        for controller in self.reflect_controllers.values():
            controller.reset_special_parameters()

        self.verify_parameter_limit()

        self.last_reflect_gate_mean = 0.0

    def forward(
        self,
        input_ids: torch.Tensor,
        targets: Optional[torch.Tensor] = None,
    ) -> ANKAOutput:
        self.validate_inputs(input_ids, targets)

        hidden_states = self.token_embedding(input_ids)

        gate_means = []

        for layer_id, layer in enumerate(self.layers):
            before = hidden_states
            proposal = layer(
                before,
                morph_router=None,
                layer_id=layer_id,
            )

            controller_key = str(layer_id)

            if controller_key in self.reflect_controllers:
                second_state = layer(
                    proposal,
                    morph_router=None,
                    layer_id=layer_id,
                )

                correction = second_state - proposal
                first_delta = proposal - before

                gate = self.reflect_controllers[
                    controller_key
                ](
                    proposal=proposal,
                    delta=first_delta,
                )

                hidden_states = (
                    proposal
                    + gate * correction
                )

                gate_means.append(
                    gate.detach().float().mean()
                )
            else:
                hidden_states = proposal

        hidden_states = self.final_norm(hidden_states)
        logits = self.compute_logits(hidden_states)
        loss = self.causal_loss(logits, targets)

        if gate_means:
            self.last_reflect_gate_mean = float(
                torch.stack(gate_means).mean().cpu()
            )

        return ANKAOutput(
            logits=logits,
            loss=loss,
        )


# ============================================================
# 3. M2 — ANKA MORPH
# ============================================================

class LowRankMorphExpert(nn.Module):
    """Shared low-rank representation transform."""

    def __init__(
        self,
        d_model: int,
        rank: int,
    ) -> None:
        super().__init__()

        self.down_projection = nn.Linear(
            d_model,
            rank,
            bias=False,
        )

        self.up_projection = nn.Linear(
            rank,
            d_model,
            bias=False,
        )

    def forward(
        self,
        x: torch.Tensor,
    ) -> torch.Tensor:
        hidden = F.silu(
            self.down_projection(x)
        )

        return self.up_projection(hidden)


class TopTwoMorphRouter(nn.Module):
    """
    Token and layer kimligine by selects experts.

    Only the two highest-probability experts remain active.
    """

    def __init__(
        self,
        config: ANKAConfig,
        router_width: int,
        num_experts: int,
    ) -> None:
        super().__init__()

        self.num_experts = num_experts

        self.norm = RMSNorm(
            config.d_model,
            eps=config.rms_norm_eps,
            affine=False,
        )

        self.summary_projection = nn.Linear(
            config.d_model,
            router_width,
            bias=False,
        )

        self.layer_embedding = nn.Embedding(
            config.n_layers,
            router_width,
        )

        self.output_projection = nn.Linear(
            router_width,
            num_experts,
            bias=True,
        )

    def forward(
        self,
        x: torch.Tensor,
        layer_id: int,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        normalized = self.norm(x)

        hidden = self.summary_projection(normalized)
        hidden = (
            hidden
            + self.layer_embedding.weight[layer_id]
        )
        hidden = F.silu(hidden)

        logits = self.output_projection(hidden)
        probabilities = F.softmax(
            logits.float(),
            dim=-1,
        )

        top_values, top_indices = torch.topk(
            probabilities,
            k=2,
            dim=-1,
        )

        sparse_weights = torch.zeros_like(
            probabilities
        )

        sparse_weights.scatter_(
            dim=-1,
            index=top_indices,
            src=top_values,
        )

        sparse_weights = sparse_weights / (
            sparse_weights.sum(
                dim=-1,
                keepdim=True,
            )
            + 1e-9
        )

        # Soft balancing penalty with a zero minimum.
        mean_probability = probabilities.mean(
            dim=(0, 1)
        )

        balance_loss = (
            self.num_experts
            * mean_probability.pow(2).sum()
            - 1.0
        )

        return (
            sparse_weights.to(dtype=x.dtype),
            balance_loss,
        )


class M2MorphModel(AdvancedANKABase):
    """
    Second-generation ANKA-Morph.

    After the standard Transformer layer runs, each token
    from four shared low-rank experts selects the two best matches.

    Expert transform:

        x → down projection → SiLU → up projection

    Experts are shared across all layers; the router receives the layer identity.
    """

    NUM_EXPERTS = 4
    EXPERT_RANK = 32
    ROUTER_WIDTH = 32
    BALANCE_COEFFICIENT = 0.01

    def __init__(
        self,
        vocab_size: int = 15_360,
        max_seq_len: int = 1_024,
    ) -> None:
        config = advanced_c2_config(
            vocab_size=vocab_size,
            max_seq_len=max_seq_len,
        )

        super().__init__(config)

        self.morph_norm = RMSNorm(
            config.d_model,
            eps=config.rms_norm_eps,
            affine=True,
        )

        self.morph_router_v2 = TopTwoMorphRouter(
            config=config,
            router_width=self.ROUTER_WIDTH,
            num_experts=self.NUM_EXPERTS,
        )

        self.morph_experts = nn.ModuleList(
            LowRankMorphExpert(
                d_model=config.d_model,
                rank=self.EXPERT_RANK,
            )
            for _ in range(self.NUM_EXPERTS)
        )

        # Each layer learns its own Morph strength.
        # Start with a small but non-zero contribution.
        self.morph_scale_logits = nn.Parameter(
            torch.full(
                (config.n_layers,),
                -4.0,
            )
        )

        self.initialize_base_parameters()

        # Prevent experts from changing model behavior abruptly at initialization.
        # to-avoid-changing for output projection'lari kucultulur.
        for expert in self.morph_experts:
            nn.init.normal_(
                expert.up_projection.weight,
                mean=0.0,
                std=1e-3,
            )

        nn.init.normal_(
            self.morph_router_v2.output_projection.weight,
            mean=0.0,
            std=1e-3,
        )

        nn.init.zeros_(
            self.morph_router_v2.output_projection.bias
        )

        self.verify_parameter_limit()

        self.last_morph_balance_loss = 0.0
        self.last_morph_scale_mean = 0.0

    def apply_morph(
        self,
        hidden_states: torch.Tensor,
        layer_id: int,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        expert_input = self.morph_norm(
            hidden_states
        )

        weights, balance_loss = (
            self.morph_router_v2(
                expert_input,
                layer_id,
            )
        )

        expert_outputs = torch.stack(
            [
                expert(expert_input)
                for expert in self.morph_experts
            ],
            dim=-2,
        )

        mixed_delta = (
            expert_outputs
            * weights.unsqueeze(-1)
        ).sum(dim=-2)

        scale = torch.sigmoid(
            self.morph_scale_logits[layer_id]
        ).to(dtype=hidden_states.dtype)

        hidden_states = (
            hidden_states
            + scale * mixed_delta
        )

        return hidden_states, balance_loss

    def forward(
        self,
        input_ids: torch.Tensor,
        targets: Optional[torch.Tensor] = None,
    ) -> ANKAOutput:
        self.validate_inputs(input_ids, targets)

        hidden_states = self.token_embedding(input_ids)
        balance_losses = []

        for layer_id, layer in enumerate(self.layers):
            hidden_states = layer(
                hidden_states,
                morph_router=None,
                layer_id=layer_id,
            )

            hidden_states, balance_loss = (
                self.apply_morph(
                    hidden_states,
                    layer_id,
                )
            )

            balance_losses.append(balance_loss)

        hidden_states = self.final_norm(hidden_states)
        logits = self.compute_logits(hidden_states)

        language_loss = self.causal_loss(
            logits,
            targets,
        )

        average_balance_loss = torch.stack(
            balance_losses
        ).mean()

        loss = language_loss

        # Validation loss contains only the true language-model loss.
        # The balancing penalty is added only during training.
        if (
            loss is not None
            and self.training
        ):
            loss = (
                loss
                + self.BALANCE_COEFFICIENT
                * average_balance_loss
            )

        self.last_morph_balance_loss = float(
            average_balance_loss.detach().cpu()
        )

        self.last_morph_scale_mean = float(
            torch.sigmoid(
                self.morph_scale_logits.detach()
            ).mean().cpu()
        )

        return ANKAOutput(
            logits=logits,
            loss=loss,
        )

## 12. Register R2 and M2 in the model factory

Defines or validates model components used by the experiment. Keep tensor shapes, parameter counts, and causal behavior unchanged when reproducing the run.


In [ ]:
from pathlib import Path

variants_path = Path(
    "/content/ANKA-50M/anka/variants.py"
)

text = variants_path.read_text(
    encoding="utf-8"
)

marker = "# ANKA_ADVANCED_VARIANTS_V2"

if marker not in text:
    addition = r'''

# ANKA_ADVANCED_VARIANTS_V2
# ============================================================
# 7. R2 / M2 SECOND-GENERATION MODELS
# ============================================================

from .advanced_variants import (
    R2ReflectModel,
    M2MorphModel,
    advanced_c2_config,
)


def r2_config(
    vocab_size: int = 15_360,
    max_seq_len: int = 1_024,
) -> ANKAConfig:
    return advanced_c2_config(
        vocab_size=vocab_size,
        max_seq_len=max_seq_len,
    )


def m2_config(
    vocab_size: int = 15_360,
    max_seq_len: int = 1_024,
) -> ANKAConfig:
    return advanced_c2_config(
        vocab_size=vocab_size,
        max_seq_len=max_seq_len,
    )


# Preserve the earlier function.
_original_experiment_config = experiment_config

EXPERIMENT_ARMS = EXPERIMENT_ARMS + (
    "R2",
    "M2",
)

MODEL_CLASSES.update(
    {
        "R2": R2ReflectModel,
        "M2": M2MorphModel,
    }
)


def experiment_config(
    name: str,
    *,
    vocab_size: int = 15_360,
    max_seq_len: int = 1_024,
) -> ANKAConfig:
    key = name.upper()

    if key == "R2":
        return r2_config(
            vocab_size=vocab_size,
            max_seq_len=max_seq_len,
        )

    if key == "M2":
        return m2_config(
            vocab_size=vocab_size,
            max_seq_len=max_seq_len,
        )

    return _original_experiment_config(
        key,
        vocab_size=vocab_size,
        max_seq_len=max_seq_len,
    )
'''

    variants_path.write_text(
        text + addition,
        encoding="utf-8",
    )

    print("R2 and M2 variants.py was added to the file.")
else:
    print("R2/M2 registration already exists; it was not added again.")

## 13. Update the shared experiment entry point

Defines or validates model components used by the experiment. Keep tensor shapes, parameter counts, and causal behavior unchanged when reproducing the run.


In [ ]:
from pathlib import Path

experiment_path = Path(
    "/content/ANKA-50M/anka/experiment.py"
)

text = experiment_path.read_text(
    encoding="utf-8"
)

old_import = (
    "from .variants import "
    "EXPERIMENT_ARMS, experiment_config"
)

new_import = (
    "from .variants import "
    "EXPERIMENT_ARMS, experiment_config, "
    "build_experiment_model"
)

if old_import in text:
    text = text.replace(
        old_import,
        new_import,
    )

old_creation = """    config = experiment_config(args.arm, vocab_size=args.vocab_size,
                               max_seq_len=max(args.sequence_length, args.max_seq_len))
    model = ANKAForCausalLM(config).to(device)"""

new_creation = """    model = build_experiment_model(
        args.arm,
        vocab_size=args.vocab_size,
        max_seq_len=max(
            args.sequence_length,
            args.max_seq_len,
        ),
    ).to(device)
    config = model.config"""

if old_creation not in text:
    raise RuntimeError(
        "Expected model-construction block was not found. "
        "experiment.py was not modified."
    )

text = text.replace(
    old_creation,
    new_creation,
)

experiment_path.write_text(
    text,
    encoding="utf-8",
)

print("experiment.py actual model siniflarini was updated to use the concrete model classes.")

## 14. Reload and structurally verify R2 and M2

Defines or validates model components used by the experiment. Keep tensor shapes, parameter counts, and causal behavior unchanged when reproducing the run.


In [ ]:
# ============================================================
# R2 / M2 CLEAN RELOAD AND TEST FROM SCRATCH
# ============================================================

%cd /content/ANKA-50M

import gc
import sys
import importlib
from pathlib import Path

import torch


# ============================================================
# 1. VERIFY REQUIRED FILES
# ============================================================

project_dir = Path("/content/ANKA-50M")
advanced_path = project_dir / "anka" / "advanced_variants.py"
variants_path = project_dir / "anka" / "variants.py"
experiment_path = project_dir / "anka" / "experiment.py"

required_files = [
    advanced_path,
    variants_path,
    experiment_path,
]

for path in required_files:
    if not path.exists():
        raise FileNotFoundError(
            f"Required file not found: {path}"
        )

print("required filelar found.")


# ============================================================
# 2. VERIFY VARIANT REGISTRATIONS
# ============================================================

variants_text = variants_path.read_text(
    encoding="utf-8"
)

experiment_text = experiment_path.read_text(
    encoding="utf-8"
)

if "ANKA_ADVANCED_VARIANTS_V2" not in variants_text:
    raise RuntimeError(
        "R2/M2 registration code variants.py filesina is missing. "
        "Run Cell 2 first."
    )

if '"R2": R2ReflectModel' not in variants_text:
    raise RuntimeError(
        "R2 MODEL_CLASSES dictionary is missing."
    )

if '"M2": M2MorphModel' not in variants_text:
    raise RuntimeError(
        "M2 MODEL_CLASSES dictionary is missing."
    )

if "build_experiment_model" not in experiment_text:
    raise RuntimeError(
        "experiment.py model factory fix is missing."
    )

print("R2/M2 file registrations verified.")


# ============================================================
# 3. RAM'DEKI REMOVE OLD ANKA MODULES
# ============================================================

modules_to_remove = [
    module_name
    for module_name in list(sys.modules)
    if (
        module_name == "anka"
        or module_name.startswith("anka.")
    )
]

for module_name in modules_to_remove:
    del sys.modules[module_name]

importlib.invalidate_caches()

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print(
    f"{len(modules_to_remove)} old ANKA modules "
    f"RAM'den temizlendi."
)


# ============================================================
# 4. IMPORT MODELS FROM SCRATCH
# ============================================================

variants_module = importlib.import_module(
    "anka.variants"
)

EXPERIMENT_ARMS = variants_module.EXPERIMENT_ARMS
MODEL_CLASSES = variants_module.MODEL_CLASSES
build_experiment_model = (
    variants_module.build_experiment_model
)

print("Experiment arms:", EXPERIMENT_ARMS)
print("Model classes:", tuple(MODEL_CLASSES.keys()))

assert "R2" in EXPERIMENT_ARMS, (
    "R2, EXPERIMENT_ARMS not found in"
)

assert "M2" in EXPERIMENT_ARMS, (
    "M2, EXPERIMENT_ARMS not found in"
)

assert "R2" in MODEL_CLASSES, (
    "R2, MODEL_CLASSES not found in"
)

assert "M2" in MODEL_CLASSES, (
    "M2, MODEL_CLASSES not found in"
)


# ============================================================
# 5. DETERMINISTIK TEST
# ============================================================

torch.manual_seed(2026)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(2026)

test_tokens = torch.randint(
    low=0,
    high=15_360,
    size=(2, 32),
    dtype=torch.long,
)


# ============================================================
# 6. C2 / R2 / M2 FORWARD TEST
# ============================================================

results = {}

for arm in ("C2", "R2", "M2"):
    print("\n" + "=" * 70)
    print(f"{arm} TESTING")
    print("=" * 70)

    model = build_experiment_model(
        arm,
        vocab_size=15_360,
        max_seq_len=128,
    )

    model.eval()

    parameter_count = (
        model.num_trainable_parameters()
    )

    assert parameter_count <= 50_000_000, (
        f"{arm} exceeds the parameter limit: "
        f"{parameter_count:,}"
    )

    with torch.no_grad():
        output = model(
            test_tokens,
            targets=test_tokens,
        )

    expected_shape = (
        2,
        32,
        15_360,
    )

    assert output.logits.shape == expected_shape, (
        f"{arm} incorrect logits shape: "
        f"{tuple(output.logits.shape)}"
    )

    assert output.loss is not None, (
        f"{arm} did not return loss"
    )

    assert torch.isfinite(output.loss), (
        f"{arm} loss is not finite: {output.loss}"
    )

    result = {
        "parameters": parameter_count,
        "logits_shape": tuple(
            output.logits.shape
        ),
        "loss": float(output.loss),
    }

    if arm == "R2":
        result["reflect_gate_mean"] = (
            model.last_reflect_gate_mean
        )

    if arm == "M2":
        result["morph_balance_loss"] = (
            model.last_morph_balance_loss
        )
        result["morph_scale_mean"] = (
            model.last_morph_scale_mean
        )

    results[arm] = result

    print(
        f"parameter : {parameter_count:,}"
    )
    print(
        f"Logits    : {tuple(output.logits.shape)}"
    )
    print(
        f"Loss      : {float(output.loss):.4f}"
    )

    if arm == "R2":
        print(
            "Mean Reflect gate:",
            f"{model.last_reflect_gate_mean:.6f}",
        )

    if arm == "M2":
        print(
            "Morph balance loss:",
            f"{model.last_morph_balance_loss:.6f}",
        )
        print(
            "Mean Morph scale:",
            f"{model.last_morph_scale_mean:.6f}",
        )

    del output
    del model
    gc.collect()


# ============================================================
# 7. RESULT
# ============================================================

print("\n" + "=" * 70)
print("ALL STRUCTURAL TESTS PASSED")
print("=" * 70)

for arm, result in results.items():
    print(
        f"{arm}: "
        f"parameters={result['parameters']:,} | "
        f"loss={result['loss']:.4f}"
    )

## 15. Train the R2 Reflect pilot

This is a compute-intensive stage. Confirm the selected runtime, input paths, token budget, and resume directory before execution. Checkpoints are written incrementally.


In [ ]:
# ============================================================
# R2 REFLECT — 256M TOKEN REAL PILOT TRAINING
# Estimated duration: 40–55 minutes
# ============================================================

%cd /content/ANKA-50M

!PYTHONUNBUFFERED=1 python -u -m anka.experiment train \
    --arm R2 \
    --manifest /content/anka_shards/manifest.json \
    --tokenizer /content/drive/MyDrive/ANKA/run_01/tokenizer/tokenizer.json \
    --output /content/drive/MyDrive/ANKA/run_01/pilot_v2 \
    --token-budget 256114688 \
    --global-batch-tokens 262144 \
    --micro-batch-size 64 \
    --eval-batch-size 8 \
    --sequence-length 1024 \
    --max-seq-len 1024 \
    --vocab-size 15360 \
    --learning-rate 3e-3 \
    --min-learning-rate 3e-4 \
    --warmup-fraction 0.05 \
    --precision bf16 \
    --device auto \
    --seed 2026 \
    --log-every 10 \
    --eval-every 100 \
    --eval-batches 20 \
    --prompts \
        "The future of" \
        "Science is important because" \
        "If a box is placed on a table" \
        "One day in Istanbul"

## 16. Train the M2 Morph pilot

This is a compute-intensive stage. Confirm the selected runtime, input paths, token budget, and resume directory before execution. Checkpoints are written incrementally.


In [ ]:
# ============================================================
# M2 MORPH — 256M TOKEN REAL PILOT TRAINING
# Estimated duration: 40–60 minutes
# ============================================================

%cd /content/ANKA-50M

!PYTHONUNBUFFERED=1 python -u -m anka.experiment train \
    --arm M2 \
    --manifest /content/anka_shards/manifest.json \
    --tokenizer /content/drive/MyDrive/ANKA/run_01/tokenizer/tokenizer.json \
    --output /content/drive/MyDrive/ANKA/run_01/pilot_v2 \
    --token-budget 256114688 \
    --global-batch-tokens 262144 \
    --micro-batch-size 64 \
    --eval-batch-size 8 \
    --sequence-length 1024 \
    --max-seq-len 1024 \
    --vocab-size 15360 \
    --learning-rate 3e-3 \
    --min-learning-rate 3e-4 \
    --warmup-fraction 0.05 \
    --precision bf16 \
    --device auto \
    --seed 2026 \
    --log-every 10 \
    --eval-every 100 \
    --eval-batches 20 \
    --prompts \
        "The future of" \
        "Science is important because" \
        "If a box is placed on a table" \
        "One day in Istanbul"

## 17. Compare C2, R2, and M2 under the matched protocol

Prepares or validates persistent data artifacts. Review the Drive paths before running; large corpus operations can take time and may replace files at the configured destination.


In [ ]:
# ============================================================
# C2 / R2 / M2 FAIR GROUP COMPARISON
# ============================================================

%cd /content/ANKA-50M

import csv
import gc
import importlib
import json
import math
import statistics
import sys
import time

from contextlib import nullcontext
from pathlib import Path

import torch


# ============================================================
# 1. SETTINGS
# ============================================================

PROJECT_DIR = Path("/content/ANKA-50M")

MANIFEST_PATH = Path(
    "/content/anka_shards/manifest.json"
)

TOKENIZER_PATH = Path(
    "/content/drive/MyDrive/ANKA/run_01/"
    "tokenizer/tokenizer.json"
)

OLD_PILOT_ROOT = Path(
    "/content/drive/MyDrive/ANKA/run_01/pilot"
)

NEW_PILOT_ROOT = Path(
    "/content/drive/MyDrive/ANKA/run_01/pilot_v2"
)

REPORT_DIR = Path(
    "/content/drive/MyDrive/ANKA/run_01/"
    "pilot_v2_comparison"
)

REPORT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

CHECKPOINTS = {
    "C2": OLD_PILOT_ROOT / "C2" / "checkpoint.pt",
    "R2": NEW_PILOT_ROOT / "R2" / "checkpoint.pt",
    "M2": NEW_PILOT_ROOT / "M2" / "checkpoint.pt",
}

SEQUENCE_LENGTH = 1024
EVAL_BATCH_SIZE = 8
EVAL_BATCHES = 100
VALIDATION_SEED = 98_761

PROMPTS = [
    "The future of",
    "Science is important because",
    "If a box is placed on a table",
    "Water freezes when",
    "One day in Istanbul",
]

MAX_NEW_TOKENS = 64

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Cihaz:", DEVICE)
print("report path:", REPORT_DIR)


# ============================================================
# 2. CHECK CHECKPOINT FILES
# ============================================================

missing_checkpoints = [
    f"{arm}: {path}"
    for arm, path in CHECKPOINTS.items()
    if not path.exists()
]

if missing_checkpoints:
    raise FileNotFoundError(
        "missing checkpoint found:\n"
        + "\n".join(missing_checkpoints)
    )

print("All checkpoint files found.")


# ============================================================
# 3. CLEAR OLD IMPORT CACHE
# ============================================================

modules_to_remove = [
    module_name
    for module_name in list(sys.modules)
    if (
        module_name == "anka"
        or module_name.startswith("anka.")
    )
]

for module_name in modules_to_remove:
    del sys.modules[module_name]

importlib.invalidate_caches()
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

variants_module = importlib.import_module(
    "anka.variants"
)

data_module = importlib.import_module(
    "anka.data"
)

build_experiment_model = (
    variants_module.build_experiment_model
)

ShardBatchIterator = (
    data_module.ShardBatchIterator
)

print(
    "Loaded experiment arms:",
    variants_module.EXPERIMENT_ARMS,
)


# ============================================================
# 4. TOKENIZER
# ============================================================

from tokenizers import Tokenizer

tokenizer = Tokenizer.from_file(
    str(TOKENIZER_PATH)
)

bos_id = tokenizer.token_to_id("<empty>")
eos_id = tokenizer.token_to_id("<eos>")

if bos_id is None:
    raise RuntimeError(
        "Tokenizer does not contain <empty>."
    )

if eos_id is None:
    raise RuntimeError(
        "Tokenizer does not contain <eos>."
    )


# ============================================================
# 5. BF16 CONTEXT
# ============================================================

def precision_context():
    if DEVICE.type == "cuda":
        return torch.autocast(
            device_type="cuda",
            dtype=torch.bfloat16,
        )

    return nullcontext()


# ============================================================
# 6. CHECKPOINT LOADING
# ============================================================

def load_model(
    arm: str,
    checkpoint_path: Path,
):
    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False,
    )

    model_config = checkpoint["model_config"]

    model = build_experiment_model(
        arm,
        vocab_size=int(
            model_config["vocab_size"]
        ),
        max_seq_len=int(
            model_config["max_seq_len"]
        ),
    )

    incompatible = model.load_state_dict(
        checkpoint["model"],
        strict=True,
    )

    if incompatible.missing_keys:
        raise RuntimeError(
            f"{arm} missing weights: "
            f"{incompatible.missing_keys}"
        )

    if incompatible.unexpected_keys:
        raise RuntimeError(
            f"{arm} unexpected weights: "
            f"{incompatible.unexpected_keys}"
        )

    model = model.to(DEVICE)
    model.eval()

    return model, checkpoint


# ============================================================
# 7. FIXED VALIDATION EVALUATION
# ============================================================

@torch.inference_mode()
def evaluate_model(
    arm: str,
    model,
) -> dict:
    validation_iterator = iter(
        ShardBatchIterator(
            MANIFEST_PATH,
            "validation",
            EVAL_BATCH_SIZE,
            SEQUENCE_LENGTH,
            VALIDATION_SEED,
        )
    )

    losses = []
    reflect_gates = []
    morph_balance_losses = []
    morph_scales = []

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()

    started = time.perf_counter()

    for _ in range(EVAL_BATCHES):
        tokens = next(
            validation_iterator
        ).to(
            DEVICE,
            dtype=torch.long,
            non_blocking=True,
        )

        with precision_context():
            output = model(
                tokens,
                targets=tokens,
            )

        loss_value = float(
            output.loss.detach().float().cpu()
        )

        if not math.isfinite(loss_value):
            raise RuntimeError(
                f"{arm} produced non-finite loss: "
                f"{loss_value}"
            )

        losses.append(loss_value)

        if arm == "R2":
            reflect_gates.append(
                float(
                    model.last_reflect_gate_mean
                )
            )

        if arm == "M2":
            morph_balance_losses.append(
                float(
                    model.last_morph_balance_loss
                )
            )

            morph_scales.append(
                float(
                    model.last_morph_scale_mean
                )
            )

    if DEVICE.type == "cuda":
        torch.cuda.synchronize()

    elapsed_seconds = (
        time.perf_counter()
        - started
    )

    prediction_tokens = (
        EVAL_BATCH_SIZE
        * (SEQUENCE_LENGTH - 1)
        * EVAL_BATCHES
    )

    average_loss = statistics.fmean(
        losses
    )

    loss_std = (
        statistics.pstdev(losses)
        if len(losses) > 1
        else 0.0
    )

    perplexity = math.exp(
        min(average_loss, 20.0)
    )

    tokens_per_second = (
        prediction_tokens
        / elapsed_seconds
    )

    if DEVICE.type == "cuda":
        peak_memory_gib = (
            torch.cuda.max_memory_allocated()
            / 2**30
        )
    else:
        peak_memory_gib = 0.0

    result = {
        "arm": arm,
        "parameters": (
            model.num_trainable_parameters()
        ),
        "validation_loss": average_loss,
        "validation_loss_std": loss_std,
        "perplexity": perplexity,
        "eval_tokens_per_second": (
            tokens_per_second
        ),
        "eval_seconds": elapsed_seconds,
        "peak_memory_gib": peak_memory_gib,
        "evaluation_batches": EVAL_BATCHES,
        "evaluation_batch_size": (
            EVAL_BATCH_SIZE
        ),
        "sequence_length": (
            SEQUENCE_LENGTH
        ),
        "validation_seed": (
            VALIDATION_SEED
        ),
    }

    if reflect_gates:
        result["reflect_gate_mean"] = (
            statistics.fmean(
                reflect_gates
            )
        )

        result["reflect_gate_min_batch"] = min(
            reflect_gates
        )

        result["reflect_gate_max_batch"] = max(
            reflect_gates
        )

    if morph_balance_losses:
        result["morph_balance_loss"] = (
            statistics.fmean(
                morph_balance_losses
            )
        )

        result["morph_scale_mean"] = (
            statistics.fmean(
                morph_scales
            )
        )

    return result


# ============================================================
# 8. GREEDY GENERATION
# ============================================================

@torch.inference_mode()
def generate_samples(
    model,
) -> list[dict]:
    samples = []

    for prompt in PROMPTS:
        encoded = tokenizer.encode(
            prompt
        ).ids

        if not encoded:
            encoded = [bos_id]

        tokens = torch.tensor(
            [
                encoded[
                    -model.config.max_seq_len:
                ]
            ],
            device=DEVICE,
            dtype=torch.long,
        )

        generated_count = 0

        for _ in range(MAX_NEW_TOKENS):
            context = tokens[
                :,
                -model.config.max_seq_len:,
            ]

            with precision_context():
                logits = model(
                    context
                ).logits[:, -1, :]

            next_token = torch.argmax(
                logits.float(),
                dim=-1,
                keepdim=True,
            )

            tokens = torch.cat(
                (
                    tokens,
                    next_token,
                ),
                dim=1,
            )

            generated_count += 1

            if int(next_token.item()) == eos_id:
                break

        decoded = tokenizer.decode(
            tokens[0].tolist(),
            skip_special_tokens=True,
        )

        samples.append(
            {
                "prompt": prompt,
                "completion": decoded,
                "generated_tokens": (
                    generated_count
                ),
            }
        )

    return samples


# ============================================================
# 9. all models in-order evaluate
# ============================================================

results = []

for arm, checkpoint_path in CHECKPOINTS.items():
    print("\n" + "=" * 80)
    print(f"{arm} EVALUATING")
    print("=" * 80)

    model, checkpoint = load_model(
        arm,
        checkpoint_path,
    )

    result = evaluate_model(
        arm,
        model,
    )

    result["training_token_budget"] = int(
        checkpoint["train_config"][
            "total_steps"
        ]
        * checkpoint["train_config"][
            "global_batch_tokens"
        ]
    )

    result["checkpoint"] = str(
        checkpoint_path
    )

    result["samples"] = generate_samples(
        model
    )

    results.append(result)

    print(
        f"Loss       : "
        f"{result['validation_loss']:.6f}"
    )

    print(
        f"Perplexity : "
        f"{result['perplexity']:.2f}"
    )

    print(
        f"Evaluation speed  : "
        f"{result['eval_tokens_per_second']:,.0f} "
        f"token/s"
    )

    print(
        f"Peak VRAM  : "
        f"{result['peak_memory_gib']:.2f} GB"
    )

    if arm == "R2":
        print(
            f"Reflect gate: "
            f"{result['reflect_gate_mean']:.6f}"
        )

    if arm == "M2":
        print(
            f"Morph balance: "
            f"{result['morph_balance_loss']:.6f}"
        )

        print(
            f"Morph scale: "
            f"{result['morph_scale_mean']:.6f}"
        )

    del model
    del checkpoint

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ============================================================
# 10. SIRALA
# ============================================================

results.sort(
    key=lambda item: (
        item["validation_loss"],
        -item["eval_tokens_per_second"],
    )
)

for ranking, result in enumerate(
    results,
    start=1,
):
    result["ranking"] = ranking

winner = results[0]["arm"]


# ============================================================
# 11. JSON report
# ============================================================

json_report = {
    "comparison": "C2_vs_R2_vs_M2",
    "selection_rule": (
        "Lowest fixed validation loss; "
        "higher evaluation throughput breaks ties"
    ),
    "winner": winner,
    "results": results,
}

json_path = (
    REPORT_DIR
    / "c2_r2_m2_comparison.json"
)

json_path.write_text(
    json.dumps(
        json_report,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)


# ============================================================
# 12. CSV report
# ============================================================

csv_path = (
    REPORT_DIR
    / "c2_r2_m2_comparison.csv"
)

csv_fields = [
    "ranking",
    "arm",
    "parameters",
    "training_token_budget",
    "validation_loss",
    "validation_loss_std",
    "perplexity",
    "eval_tokens_per_second",
    "eval_seconds",
    "peak_memory_gib",
    "reflect_gate_mean",
    "morph_balance_loss",
    "morph_scale_mean",
    "checkpoint",
]

with csv_path.open(
    "w",
    newline="",
    encoding="utf-8",
) as handle:
    writer = csv.DictWriter(
        handle,
        fieldnames=csv_fields,
        extrasaction="ignore",
    )

    writer.writeheader()

    for result in results:
        writer.writerow(result)


# ============================================================
# 13. RESULT EKRANI
# ============================================================

print("\n" + "=" * 90)
print("ANKA-50M C2 / R2 / M2 comparison")
print("=" * 90)

for result in results:
    extra = ""

    if result["arm"] == "R2":
        extra = (
            f" | reflect="
            f"{result['reflect_gate_mean']:.4f}"
        )

    if result["arm"] == "M2":
        extra = (
            f" | morph_scale="
            f"{result['morph_scale_mean']:.4f}"
            f" | balance="
            f"{result['morph_balance_loss']:.6f}"
        )

    print(
        f"{result['ranking']}. "
        f"{result['arm']} | "
        f"loss={result['validation_loss']:.6f} | "
        f"ppl={result['perplexity']:.2f} | "
        f"eval={result['eval_tokens_per_second']:,.0f} tok/s | "
        f"VRAM={result['peak_memory_gib']:.2f} GB"
        f"{extra}"
    )

print("\nKazanan:", winner)

print("\nRaporlar saved:")
print(json_path)
print(csv_path)